# 🏅 Evaluation & Leaderboard

Scores the submissions and produces the leaderboard. **Organisers only**, because this reads
the gold labels, which participants do not have.

On the day each person writes into their own folder on the workshop volume. This notebook
reads those folders, scores everything it finds, and ranks it.

## How it works

1. **Find** the submissions. On `azure`, one folder per person under `output/`, taking the
   newest `.csv` from each and skipping `GoldData` and `Submissions`. Locally and on GCS,
   every `.csv` in a single shared folder.
2. **Identify** each one. On `azure` a submission is its folder plus its file name, for
   example `user-02-team1`, so two people on one team get a row each whether they worked
   together or separately. Locally and on GCS the file must be called `teamN.csv`, and
   anything else is listed as rejected with the reason.
3. **Keep the newest.** Only the most recent file counts, so re-submitting simply replaces
   the previous attempt.
4. **Validate** each submission: required columns, duplicate compounds, unknown SMILES.
5. **Score** it against the gold labels: hits, precision and enrichment at several depths.
6. **Rank** and draw the leaderboard.

Re-run the whole notebook to refresh. For a live leaderboard, run it on a timer. Every five
minutes is usually enough.

## Configuration

The only cell you normally touch. `SOURCE` switches between reading a local folder and reading
a GCS bucket; everything downstream is identical either way.

In [ ]:
# Make sure the packages this notebook needs are present.
# pyarrow is what pandas uses to read the gold-label parquet, and it is easy to
# forget because Colab and Databricks both happen to ship it.
# gcsfs is not installed here - it is only needed when SOURCE = "gcp", and the
# cell that reads the bucket says so if it is missing.
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("matplotlib", "matplotlib"),
]

missing = [pkg for module, pkg in REQUIRED if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done.")
else:
    print("All required packages are already available.")

In [ ]:
import os
import re
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- where to read from ----------------------------------------------------
SOURCE = "local"          # "local", "azure" or "gcp"

# local: the folder participants write into, and the labelled parquet
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

LOCAL_SUBMISSIONS = REPO_ROOT / "team-results"
LOCAL_GOLD = REPO_ROOT / "sample_data" / "sample-test.parquet"

# azure: one folder per user on the workshop volume. Each person writes their own CSVs into
# their own folder, so a submission is identified by the folder it came from plus the file
# name, for example "user-02-team1". Two people on one team therefore get a row each,
# whether they work together or separately.
AZURE_OUTPUT = Path("/Volumes/uhn_workshop/lab/output")
AZURE_GOLD = AZURE_OUTPUT / "GoldData" / "Gold.parquet"
IGNORE_FOLDERS = {"GoldData", "Submissions"}

# gcp: teams have write-only access to the first, nobody but us can read the second
GCP_SUBMISSIONS = "gs://aircheck-workshop-writeonly/results"
GCP_GOLD = "gs://aircheck-workshop-readonly/WithLabels_TestDataset_Aircheck.csv"
GCP_TOKEN = None          # None uses your default credentials; "anon" for public buckets

# --- what a submission looks like ------------------------------------------
SMILES_COLUMN = "SMILES"
SCORE_COLUMN = "Prediction_Score"
LABEL_COLUMN = "LABEL"    # in the gold file
EXPECTED_ROWS = 200

# only teamN.csv is accepted, N being a whole number
TEAM_PATTERN = re.compile(r"^team(\d+)$", re.IGNORECASE)

# --- scoring ---------------------------------------------------------------
KS = (20, 50, 100, 200)
# leaderboard order: top-heavy, because a screen only ever tests the top of the list
RANK_BY = ["Hit@20", "Hit@50", "Hit@100", "Hit@200"]

print(f"Source          : {SOURCE}")
print(f"Repository root : {REPO_ROOT}")
SUBMISSION_SOURCE = {"local": LOCAL_SUBMISSIONS, "azure": AZURE_OUTPUT, "gcp": GCP_SUBMISSIONS}[SOURCE]
GOLD_SOURCE = {"local": LOCAL_GOLD, "azure": AZURE_GOLD, "gcp": GCP_GOLD}[SOURCE]

print(f"Submissions     : {SUBMISSION_SOURCE}")
print(f"Gold labels     : {GOLD_SOURCE}")

## Reading the submission folder

One pair of helpers hides the difference between a local directory and a bucket, so nothing
below this point cares which you chose. `gcsfs` is imported only when it is actually needed.

In [ ]:
def looks_like_submission(path):
    """Does this csv carry the two columns a submission needs? Header only, so it is cheap."""
    try:
        header = pd.read_csv(path, nrows=0).columns
    except Exception:
        return False
    return SMILES_COLUMN in header and SCORE_COLUMN in header


def newest_csv(folder):
    """The newest .csv in one folder that looks like a submission, or None.

    Scratch files, notes and intermediate data that someone happens to save into their own
    folder are skipped rather than taken as their entry, which would otherwise hide the real
    submission simply for being older.
    """
    csvs = sorted((p for p in Path(folder).glob("*.csv") if p.is_file()),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    for path in csvs:
        if looks_like_submission(path):
            return path
    return None


def modified_utc(path):
    return datetime.fromtimestamp(Path(path).stat().st_mtime, tz=timezone.utc)


def team_from_filename(stem):
    """Local and GCS submissions are identified by a teamN file name."""
    match = TEAM_PATTERN.match(stem)
    return f"team{int(match.group(1))}" if match else None


def list_submission_files():
    """Every submission we can see, with who sent it and when it was last modified (UTC).

    On azure each person has their own output folder, so we take the newest CSV from each
    one and identify it by folder plus file name. Locally and on GCS everything sits in a
    single folder and the teamN file name is the identity.
    """
    found = []

    if SOURCE == "gcp":
        try:
            import gcsfs
        except ImportError:
            raise ImportError("Reading from GCS needs gcsfs:  pip install gcsfs")

        fs = gcsfs.GCSFileSystem(token=GCP_TOKEN)
        for path in fs.ls(GCP_SUBMISSIONS):
            if not path.endswith(".csv"):
                continue
            info = fs.info(path)
            modified = info.get("updated") or info.get("timeCreated")
            if isinstance(modified, str):
                modified = pd.to_datetime(modified).to_pydatetime()
            stem = Path(path).stem
            found.append({"path": "gs://" + path.lstrip("/"),
                          "name": stem,
                          "user": "",
                          "team": team_from_filename(stem),
                          "modified": modified})

    elif SOURCE == "azure":
        folders = sorted(p for p in AZURE_OUTPUT.iterdir() if p.is_dir())
        empty = []
        for folder in folders:
            if folder.name in IGNORE_FOLDERS:
                continue
            newest = newest_csv(folder)
            if newest is None:
                empty.append(folder.name)
                continue
            found.append({"path": str(newest),
                          "name": newest.stem,
                          "user": folder.name,
                          "team": f"{folder.name}-{newest.stem}",
                          "modified": modified_utc(newest)})
        if empty:
            print(f"{len(empty)} folder(s) with no csv yet: {', '.join(empty)}")

    else:
        for path in sorted(Path(LOCAL_SUBMISSIONS).glob("*.csv")):
            found.append({"path": str(path),
                          "name": path.stem,
                          "user": "",
                          "team": team_from_filename(path.stem),
                          "modified": modified_utc(path)})

    return found


def read_submission(path):
    """Read one submission, from disk, from a volume, or from a bucket."""
    if str(path).startswith("gs://"):
        return pd.read_csv(path, storage_options={"token": GCP_TOKEN})
    return pd.read_csv(path)


files = list_submission_files()
print(f"Found {len(files)} csv file(s).")
for f in sorted(files, key=lambda f: f["modified"], reverse=True):
    who = f"{f['user']:<10}" if f["user"] else ""
    print(f"  {who}{f['name']:<20} {f['modified']:%Y-%m-%d %H:%M:%S} UTC")

## Which files count

On `azure` every folder under `output/` is treated as a submitter apart from `GoldData` and
`Submissions`, and the newest `.csv` inside each folder is that person's entry. Locally and
on GCS a file is scored only if it is named `teamN.csv`. Anything rejected is reported rather
than silently ignored, so on the day you can tell someone *why* their submission is missing
from the board.

Where there is more than one file, the **most recent** wins. That makes re-submitting the
obvious thing it should be: your latest attempt is your entry.

In [ ]:
def natural_key(name):
    """Sort user-2 before user-10, and team2 before team10."""
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]


accepted, rejected = {}, []

for f in files:
    team = f.get("team")
    if not team:
        rejected.append({**f, "reason": "name is not teamN"})
        continue

    previous = accepted.get(team)
    if previous is None:
        accepted[team] = f
    elif f["modified"] > previous["modified"]:
        accepted[team] = f
        rejected.append({**previous, "reason": f"superseded by a newer {team} file"})
    else:
        rejected.append({**f, "reason": f"older than another {team} file"})

accepted = dict(sorted(accepted.items(), key=lambda kv: natural_key(kv[0])))

print(f"Accepted {len(accepted)} submission(s):")
for team, f in accepted.items():
    print(f"  {team:<24} {Path(f['path']).name:<20} {f['modified']:%Y-%m-%d %H:%M} UTC")

if rejected:
    print(f"{chr(10)}Rejected {len(rejected)} file(s):")
    for f in rejected:
        print(f"  {Path(f['path']).name:<20} {f['reason']}")
else:
    print(f"{chr(10)}No files rejected.")

## The gold labels

The answer key: which compounds in the screening library are actually active. Locally this
comes from the labelled parquet; on the day it is a CSV in the read-only bucket.

In [ ]:
if SOURCE == "gcp":
    gold_df = pd.read_csv(GCP_GOLD, storage_options={"token": GCP_TOKEN})
elif SOURCE == "azure":
    gold_df = (pd.read_parquet(AZURE_GOLD) if str(AZURE_GOLD).endswith(".parquet")
               else pd.read_csv(AZURE_GOLD))
else:
    gold_df = pd.read_parquet(LOCAL_GOLD)[[SMILES_COLUMN, LABEL_COLUMN]]

gold_df = gold_df[[SMILES_COLUMN, LABEL_COLUMN]].drop_duplicates(subset=SMILES_COLUMN)
gold = dict(zip(gold_df[SMILES_COLUMN], gold_df[LABEL_COLUMN].astype(int)))

n_library = len(gold)
n_actives = int(sum(gold.values()))
base_rate = n_actives / n_library

print(f"Library   : {n_library:,} compounds")
print(f"Actives   : {n_actives} ({base_rate:.3%})")
print(f"Random pick of 200 would find about {200 * base_rate:.1f} of them.")

## Scoring

For each submission, at several depths **K**:

| | |
|---|---|
| **Hit@K** | actives among the top K, the headline number |
| **Precision@K** | Hit@K ÷ K, the same thing as a rate |
| **Enrichment@K** | how many times better than picking K at random |

Submissions are also checked as they are read. Problems are recorded rather than raised, so
one broken file cannot take the leaderboard down mid-hackathon.

In [ ]:
def score_submission(team, path):
    """Score one file. Returns (row, notes) - notes lists anything odd about it."""
    notes = []
    df = read_submission(path)

    missing = [c for c in (SMILES_COLUMN, SCORE_COLUMN) if c not in df.columns]
    if missing:
        return None, [f"missing column(s): {', '.join(missing)}"]

    df = df[[SMILES_COLUMN, SCORE_COLUMN]].copy()
    df[SCORE_COLUMN] = pd.to_numeric(df[SCORE_COLUMN], errors="coerce")

    if df[SCORE_COLUMN].isna().any():
        notes.append(f"{int(df[SCORE_COLUMN].isna().sum())} non-numeric score(s) dropped")
        df = df.dropna(subset=[SCORE_COLUMN])

    # rank by score ourselves, so a file saved in the wrong order is still scored fairly
    if not df[SCORE_COLUMN].is_monotonic_decreasing:
        notes.append("rows were not best-first; re-sorted by score")
    df = df.sort_values(SCORE_COLUMN, ascending=False, kind="stable")

    duplicated = int(df[SMILES_COLUMN].duplicated().sum())
    if duplicated:
        notes.append(f"{duplicated} duplicate compound(s), kept the highest-scoring copy")
        df = df.drop_duplicates(subset=SMILES_COLUMN, keep="first")

    if len(df) != EXPECTED_ROWS:
        notes.append(f"{len(df)} rows, expected {EXPECTED_ROWS}")

    known = df[SMILES_COLUMN].isin(gold)
    if not known.all():
        notes.append(f"{int((~known).sum())} compound(s) not in the library, ignored")
        df = df[known]

    labels = df[SMILES_COLUMN].map(gold).to_numpy()

    row = {"Team": team, "Rows": len(df)}
    for k in KS:
        hits = int(labels[:k].sum())
        row[f"Hit@{k}"] = hits
        row[f"Precision@{k}"] = round(hits / min(k, len(labels)), 4) if len(labels) else 0.0
        row[f"Enrichment@{k}"] = (round((hits / min(k, len(labels))) / base_rate, 1)
                                  if len(labels) and base_rate else float("nan"))
    row["Total hits"] = int(labels.sum())
    return row, notes


rows, all_notes = [], {}
for team, f in accepted.items():
    row, notes = score_submission(team, f["path"])
    if row is None:
        all_notes[team] = notes
        print(f"  {team:<10} NOT SCORED - {notes[0]}")
        continue
    row["Submitted"] = f["modified"].strftime("%H:%M:%S")
    rows.append(row)
    if notes:
        all_notes[team] = notes

scores_df = pd.DataFrame(rows)
if scores_df.empty:
    # Keep the columns even with no rows, so the leaderboard and charts below still run.
    columns = ["Team", "Rows"]
    for k in KS:
        columns += [f"Hit@{k}", f"Precision@{k}", f"Enrichment@{k}"]
    scores_df = pd.DataFrame(columns=columns + ["Total hits", "Submitted"])

print(f"{chr(10)}Scored {len(scores_df)} submission(s).")

if all_notes:
    print(f"{chr(10)}Submission warnings:")
    for team, notes in all_notes.items():
        for n in notes:
            print(f"  {team:<10} {n}")

## The leaderboard

Ranked by `RANK_BY`, by default Hit@20 first, falling back to deeper cuts to break ties. A
screening campaign only ever assays the top of the list, so a submission that puts actives in
its first twenty beats one that merely gets them into the first two hundred.

In [ ]:
leaderboard = (scores_df
               .sort_values(RANK_BY, ascending=[False] * len(RANK_BY))
               .reset_index(drop=True))

medals = ["🥇", "🥈", "🥉"] + [""] * max(0, len(leaderboard) - 3)
leaderboard.insert(0, "Rank", [f"{i + 1} {m}".strip()
                               for i, m in enumerate(medals[:len(leaderboard)])])

display_columns = (["Rank", "Team"]
                   + [f"Hit@{k}" for k in KS]
                   + ["Total hits", f"Enrichment@{KS[0]}", "Submitted"])

print("=" * 78)
print(f"LEADERBOARD    {datetime.now():%Y-%m-%d %H:%M:%S}    "
      f"{n_actives} actives among {n_library:,} compounds")
print("=" * 78)
print(leaderboard[display_columns].to_string(index=False))

if not leaderboard.empty:
    best = leaderboard.iloc[0]
    print(f"{chr(10)}Leading: {best['Team']} with {best[RANK_BY[0]]} "
          f"in the top {KS[0]} ({best[f'Enrichment@{KS[0]}']}x better than random).")

## Charts

Two views. The bars show how many actives each team found at each depth. The curves show
*where* in the ranking those actives turned up. A team whose curve rises early is putting
its best guesses first, which is the whole game.

In [ ]:
order = leaderboard["Team"].tolist()

fig, ax = plt.subplots(figsize=(11, 4.5))
width = 0.8 / len(KS)
x = np.arange(len(order))
palette = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

for i, k in enumerate(KS):
    values = [leaderboard.loc[leaderboard["Team"] == t, f"Hit@{k}"].iloc[0] for t in order]
    ax.bar(x + i * width - 0.4 + width / 2, values, width=width,
           label=f"Hit@{k}", color=palette[i % len(palette)])

ax.axhline(n_actives, color="grey", linestyle="--", linewidth=1,
           label=f"all {n_actives} actives")
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_ylabel("actives found")
ax.set_ylim(0, n_actives * 1.12)          # headroom so the legend clears the bars
ax.set_title("Actives found at each depth", fontweight="bold", pad=30)
ax.legend(ncol=len(KS) + 1, fontsize=9, frameon=False,
          loc="lower center", bbox_to_anchor=(0.5, 1.01))
fig.tight_layout()
plt.show()

In [ ]:
# How quickly does each team accumulate hits as you read down their list?
fig, ax = plt.subplots(figsize=(9, 5))

for team in order:
    path = accepted[team]["path"]
    df = read_submission(path)
    df = df[[SMILES_COLUMN, SCORE_COLUMN]].dropna()
    df = df.sort_values(SCORE_COLUMN, ascending=False, kind="stable")
    df = df.drop_duplicates(subset=SMILES_COLUMN, keep="first")
    labels = df[SMILES_COLUMN].map(gold).fillna(0).to_numpy()
    ax.step(np.arange(1, len(labels) + 1), np.cumsum(labels), where="post", label=team)

depth = max(KS)
ax.plot([0, depth], [0, depth * base_rate], color="grey", linestyle="--",
        linewidth=1, label="random")
ax.set_xlim(0, depth)
ax.set_xlabel("compounds tested, working down the submitted ranking")
ax.set_ylabel("actives found so far")
ax.set_title("Where in each ranking the hits appear", fontweight="bold")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## Which actives did anyone find?

Useful at the debrief. A compound everyone found was easy; one that a single team found is
worth asking about; one nobody found is worth showing on the closing slide.

In [ ]:
active_smiles = [s for s, lab in gold.items() if lab == 1]
found = pd.DataFrame(0, index=active_smiles, columns=order, dtype=int)

for team in order:
    df = read_submission(accepted[team]["path"])
    picked = set(df[SMILES_COLUMN].dropna())
    for smiles in active_smiles:
        found.loc[smiles, team] = int(smiles in picked)

found["teams that found it"] = found.sum(axis=1)
found = found.sort_values("teams that found it", ascending=False)

short = found.copy()
short.index = [s[:44] + ("…" if len(s) > 44 else "") for s in short.index]

print(f"Each of the {len(active_smiles)} actives, and who nominated it:{chr(10)}")
print(short.to_string())

never = int((found["teams that found it"] == 0).sum())
everyone = int((found["teams that found it"] == len(order)).sum())
print(f"{chr(10)}Found by every team : {everyone}")
print(f"Found by nobody     : {never}")

## Saving the board

Writes the leaderboard beside the submissions so it can be picked up by whatever is displaying
it. Skip this cell if you are only looking.

In [ ]:
SAVE_LEADERBOARD = True

if SAVE_LEADERBOARD and not leaderboard.empty:
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    out_dir = REPO_ROOT / "results"
    out_dir.mkdir(exist_ok=True)

    latest = out_dir / "leaderboard.csv"
    leaderboard.to_csv(latest, index=False)
    leaderboard.to_csv(out_dir / f"leaderboard-{stamp}.csv", index=False)

    print(f"Wrote {latest}")
    print(f"Wrote {out_dir / f'leaderboard-{stamp}.csv'}")
else:
    print("Not saved.")